In [13]:
# To install packages in Jupyter notebook, use the following syntax:
# Use ! or % to run shell commands

# Torch
!pip install torch

# OpenCV
!pip install opencv-python

# Rasterio
!pip install rasterio

# Shapely-2.1.2 imgaug-0.4.0
!pip install imgaug

#!pip install imgaug.augmenters

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [14]:
import os
# Change the working directory to the appropriate path
# If using Stoner_lab One Drive files, UPDATE "C:\Users\mallo\" with local synced path
os.chdir(r"C:\Users\mallo\OneDrive - University of Arizona\Stoner_lab - Documents\Stoner Lab Home\Projects\Automated Count Project\bighatchet")

# Check your current working directory
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

Current working directory: C:\Users\mallo\OneDrive - University of Arizona\Stoner_lab - Documents\Stoner Lab Home\Projects\Automated Count Project\bighatchet


In [15]:
from queue import Queue
from threading import Thread
import sys
import time
import cv2

class VideoReader:
    def __init__(self, video_file, queue_size=128):
        # initialize the file video stream along with the boolean
        # used to indicate if the thread should be stopped or not
        self.stream = cv2.VideoCapture(video_file)
        self.stopped = False
        # initialize the queue used to store frames read from
        # the video file
        self.Q = Queue(maxsize=queue_size)
        self.frame_num = 0
        self.false_count = 0 
        print('reading video from ', video_file)
        


    def start(self):
        # start a thread to read frames from the file video stream
        t = Thread(target=self.update, args=())
        t.daemon = True
        t.start()
        time.sleep(5)
        print('starting video reader...')
        return self

    def update(self):
        # keep looping infinitely
#         frame_num = 0
        print('update starting...')
        while True:
            if frame_num == 100:
                break
            # if the thread indicator variable is set, stop the
            # thread
            if self.stopped:
                return

            # otherwise, ensure the queue has room in it
            if not self.Q.full():
                # read the next frame from the file

                (grabbed, frame) = self.stream.read()
                self.frame_num += 1
                
                
                # add the frame to the queue
                if not grabbed:
                    if self.false_count > 500:
                        print('no more frames')
                        self.stop()
                        return
                    else:
                        if self.frame_num % 25 == 0:
                            print('queue, {}, {}'.format(self.frame_num, grabbed))
                        self.false_count += 1
                else:
                    self.Q.put(frame)
                    self.false_count = 0
                

    def read(self):
        # return next frame in the queue
        return self.Q.get()


    def more(self):
        # return True if there are still frames in the queue
        return self.Q.qsize() > 0

    def stop(self):
        # indicate that the thread should be stopped
        self.stopped = True
        self.stream.release()
        print('stopped')
    
    def is_stopped(self):
        return self.stopped
    
    def is_full(self):
        return self.Q.full()
        
        
    def close(self):
        self.stop()

In [16]:
# import glob
# parent_video_folder = '.../kasanka-bats/gopros' 

# video_files = []
# for date in ['17Nov']:
# #     video_files.extend(glob.glob(parent_video_folder + date + '/*/*.MP4'))
#     video_files.extend(glob.glob(parent_video_folder + date + '/card-f/*.MP4'))
# video_files.sort()
#     # vf = glob.glob(parent_video_folder + '/14_11_g/*.MP4')
#     # vf.sort()
#     # video_files.extend(vf)
# print(*video_files, sep='\n')

In [17]:
import glob
import os
parent_video_folder = f'bighatchet-bats\\cameras'

video_files = []
for date in ['14Jul']:
    # video_files.extend(glob.glob(parent_video_folder + date + '/*/*.MP4'))
    # video_files.extend(glob.glob(parent_video_folder + date + '/card-f/*.MP4'))
    # if you want to find all MP4 files regardless of subdirectory
    video_files.extend(glob.glob(os.path.join(parent_video_folder, date, '**', '*.MP4'), recursive=True))
video_files.sort()
    # vf = glob.glob(parent_video_folder + '/14_11_g/*.MP4')
    # vf.sort()
    # video_files.extend(vf)
print(*video_files, sep='\n')

bighatchet-bats\cameras\14Jul\thermal\14Jul_thermal.mp4


In [18]:
# import glob
# import os
# from multiprocessing import Pool
# skip_existing_folders = False
# save_frames = True
# process_every_n_frames = 1

# # existing_image_folders = glob.glob(output_folder + '/*')
# # existing_folder_names = [os.path.basename(image_folder) for image_folder in existing_image_folders]


# def pull_out_frames(video_file):
    
#    # parent_output_folder = '.../kasanka-bats/frames/'
#     parent_output_folder = f'bighatchet-bats\\frames' # Directory path for output frames
#    # parent_output_folder = '.../Elements/bats'
#     parent_output_folder = f'Elements\\bats'

#     output_folder = os.path.join(parent_output_folder, video_file.split('/')[-3], video_file.split('/')[-2])
    
#     observation_name = video_file.split('/')[-2]

#     video_name = os.path.basename(video_file)
#     video_name = video_name.split('.')[-2]

#     if skip_existing_folders:
#         if video_name in existing_folder_names:
#             return

#     print('Processing', video_name)

#     if save_frames:
        
#         output_folder = os.path.join(output_folder, video_name)

#         if not os.path.exists(output_folder):
#             os.makedirs(output_folder)

#         print('saving images to ', output_folder)
#     else:
#         print('frames are not being saved')





#     frame_count = 0
#     video_stream = VideoReader(video_file).start()
#     time.sleep(1)
#     last_frame = None
    


#     while video_stream.more():
        
        


#         if video_stream.is_full() or video_stream.is_stopped():

#             if frame_count % 4000 == 0:
#                 print(frame_count, 'frames processed')

#             frame = video_stream.read()

#             if save_frames:
#                 if frame_count % process_every_n_frames == 0:
#                     frame_name = "{}_{:05d}".format(os.path.splitext(video_name)[0], frame_count)
#                     frame_file = os.path.join(output_folder, frame_name + '.jpg')
#                     cv2.imwrite(frame_file, frame)
                    


#             frame_count += 1

#     video_stream.stop()
    

# if __name__ == '__main__':


#     with Pool(processes=4) as pool:
#         pool.map(pull_out_frames, video_files[2:])
#         # connected_distances, connected_sizes = find_tracks_in_clips(file_dicts[0])

In [19]:
import glob
import os
from multiprocessing import Pool
skip_existing_folders = False
save_frames = True
process_every_n_frames = 1

def pull_out_frames(video_file):
    
    # Directory path for output frames
    parent_output_folder = f'Elements\\bats'

    video_path_parts = os.path.normpath(video_file).split(os.sep) #works with \ 
    output_folder = os.path.join(parent_output_folder, video_path_parts[-3], video_path_parts[-2])
    observation_name = video_path_parts[-2]

    video_name = os.path.basename(video_file)
    video_name = os.path.splitext(video_name)[0] # Better way to get filename without extension

    if skip_existing_folders:
        # Define existing folders at the appropriate scope
        existing_image_folders = glob.glob(os.path.join(output_folder, '*'))
        existing_folder_names = [os.path.basename(image_folder) for image_folder in existing_image_folders]
        
        if video_name in existing_folder_names:
            print(f"Skipping {video_name} - folder already exists")
            return []  # Return empty list if skipping

    print('Processing', video_name)

    if save_frames:
        output_folder = os.path.join(output_folder, video_name)

        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        print('saving images to ', output_folder)
    else:
        print('frames are not being saved')

    frame_count = 0
    video_stream = VideoReader(video_file).start()
    time.sleep(1)
    last_frame = None
    
    frame_files = []  # Create a list to store all frame file paths
    
    while video_stream.more():
        if video_stream.is_full() or video_stream.is_stopped():
            if frame_count % 4000 == 0:
                print(frame_count, 'frames processed')

            frame = video_stream.read()

            if save_frames:
                if frame_count % process_every_n_frames == 0:
                    frame_name = "{}_{:05d}".format(os.path.splitext(video_name)[0], frame_count)
                    frame_file = os.path.join(output_folder, frame_name + '.jpg')
                    cv2.imwrite(frame_file, frame)
                    frame_files.append(frame_file)  # Add path to list

            frame_count += 1

    video_stream.stop()
    return frame_files  # Return the list of frame files - moved to end of function

if __name__ == '__main__':
    with Pool(processes=4) as pool:
        results = pool.map(pull_out_frames, video_files[2:])
        # results will be a list of lists containing all frame file paths
        print(results)

[]


In [20]:
frame_file

NameError: name 'frame_file' is not defined

In [21]:
video_files

['bighatchet-bats\\cameras\\14Jul\\thermal\\14Jul_thermal.mp4']

In [22]:
# import glob 
# import cv2
# parent_video_folder = '/Volumes/TTT_Paris/bats/video/14_11/14_11_d'

#video_files = glob.glob(parent_video_folder + '/*.MP4')
# Use os.path.join for path construction instead of string concatenation
# stream = cv2.VideoCapture(video_files[0])
# for i in range(300):
#     (grabbed, frame) = stream.read()
#     print(i, grabbed)

In [23]:
import glob 
import cv2

# parent_video_folder = '/Volumes/TTT_Paris/bats/video/14_11/14_11_d'
parent_video_folder = 'bighatchet-bats\\cameras\\14Jul\\thermal\\'

video_files = glob.glob(parent_video_folder + '/*.MP4')

stream = cv2.VideoCapture(video_files[0])

for i in range(300):
    (grabbed, frame) = stream.read()
    print(i, grabbed)

0 True
1 True
2 True
3 True
4 True
5 True
6 True
7 True
8 True
9 True
10 True
11 True
12 True
13 True
14 True
15 True
16 True
17 True
18 True
19 True
20 True
21 True
22 True
23 True
24 True
25 True
26 True
27 True
28 True
29 True
30 True
31 True
32 True
33 True
34 True
35 True
36 True
37 True
38 True
39 True
40 True
41 True
42 True
43 True
44 True
45 True
46 True
47 True
48 True
49 True
50 True
51 True
52 True
53 True
54 True
55 True
56 True
57 True
58 True
59 True
60 True
61 True
62 True
63 True
64 True
65 True
66 True
67 True
68 True
69 True
70 True
71 True
72 True
73 True
74 True
75 True
76 True
77 True
78 True
79 True
80 True
81 True
82 True
83 True
84 True
85 True
86 True
87 True
88 True
89 True
90 True
91 True
92 True
93 True
94 True
95 True
96 True
97 True
98 True
99 True
100 True
101 True
102 True
103 True
104 True
105 True
106 True
107 True
108 True
109 True
110 True
111 True
112 True
113 True
114 True
115 True
116 True
117 True
118 True
119 True
120 True
121 True
122 True
123

In [24]:
# import glob 
# import cv2
# import os

# # Use a raw string for Windows path
# parent_video_folder = r"C:\Users\mallo\OneDrive - University of Arizona\Stoner_lab - Documents\Stoner Lab Home\Projects\Automated Count Project\bighatchet\bighatchet-bats\cameras\20Jul\thermal"

# # Use os.path.join for path construction instead of string concatenation
# video_files = glob.glob(os.path.join(parent_video_folder, '*.MP4'))

# # Add error handling in case no files are found
# if not video_files:
#     print(f"No MP4 files found in {parent_video_folder}")
# else:
#     print(f"Found {len(video_files)} MP4 files")
    
#     # Try to open the first video file
#     stream = cv2.VideoCapture(video_files[0])
    
#     if not stream.isOpened():
#         print(f"Error: Could not open video file {video_files[0]}")
#     else:
#         for i in range(300):
#             (grabbed, frame) = stream.read()
#             if not grabbed:
#                 print(f"End of video reached at frame {i}")
#                 break
#             print(f"Frame {i}: successfully grabbed")
        
#         # Don't forget to release the video capture object
#         stream.release()